# Salon Experiment & Causal Analysis
**Yuva Intern project | Synthetic dataset**

> All data are generated for demonstration. They are not real salon/customer records.

## Setup and files
Keep this notebook in the same folder as the CSV files, or update `DATA_DIR`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import statsmodels.formula.api as smf

DATA_DIR = "."  # Change if your CSV files are stored elsewhere
appointments = pd.read_csv(f"{DATA_DIR}/appointments_300k.csv", parse_dates=["appointment_date"])
experiment = pd.read_csv(f"{DATA_DIR}/experiment_cohort_8000.csv")
products = pd.read_csv(f"{DATA_DIR}/product_sales_65k.csv", parse_dates=["sale_date"])


## 1. Data validation and cleaning

In [ ]:
print(appointments.shape)
print(appointments.isna().sum().sort_values(ascending=False).head())
assert appointments["appointment_id"].is_unique
assert appointments["outlet_id"].nunique() == 22
assert appointments["appointment_id"].notna().all()
print("Walk-in rows without client IDs:", appointments["client_id"].isna().sum())
print("Raw service labels:", appointments["service_name_raw"].nunique())
print("Standardized services:", appointments["service_name_standard"].nunique())


## 2. Exploratory analysis

In [ ]:
appointments["month"] = appointments["appointment_date"].dt.to_period("M").astype(str)
monthly = appointments.groupby("month").agg(
    appointments=("appointment_id","count"),
    completion_rate=("completed_flag","mean"),
    no_show_rate=("no_show_flag","mean"),
    revenue_per_scheduled_appointment=("realized_revenue_inr","mean")
).reset_index()
display(monthly.tail())
monthly.plot(x="month", y=["completion_rate","no_show_rate"], marker="o", figsize=(10,4))
plt.xticks(rotation=45); plt.tight_layout()


## 3. A/B test design and power analysis

- Unit: eligible identifiable client, randomized 1:1 within outlet and client-status strata.
- Control: standard reminder.
- Treatment: 48-hour WhatsApp reminder, rebooking link and return-visit offer.
- Primary metric: completed repeat visit within 60 days.
- Guardrails: revenue per completed visit, no-show rate and contribution margin.
- Analyze by intention-to-treat.


In [ ]:
baseline_retention = 0.34
target_retention = 0.39
alpha = 0.05
power = 0.80
effect = proportion_effectsize(baseline_retention, target_retention)
n_per_arm = int(np.ceil(NormalIndPower().solve_power(
    effect_size=abs(effect), power=power, alpha=alpha, ratio=1,
    alternative="two-sided"
)))
print("Required clients per arm:", n_per_arm)
print("Total required:", 2*n_per_arm)
eligible_per_week = 1500
print("Approximate recruitment weeks:", int(np.ceil(2*n_per_arm/eligible_per_week)))
print("Add 60 days for outcome maturation.")


## 4. Randomization balance and sample-ratio mismatch

In [ ]:
t = experiment[experiment.experiment_group == "Treatment"]
c = experiment[experiment.experiment_group == "Control"]
n_t, n_c = len(t), len(c)
srm_p = stats.binomtest(n_t, n=n_t+n_c, p=0.5).pvalue
print({"treatment_n":n_t, "control_n":n_c, "SRM_p":srm_p})
for col in ["age","prior_visits_90d","prior_revenue_90d_inr"]:
    result = stats.ttest_ind(t[col], c[col], equal_var=False)
    print(col, "means:", round(t[col].mean(),2), round(c[col].mean(),2), "p:", result.pvalue)
print(pd.crosstab(experiment.experiment_group, experiment.new_client_flag, normalize="index"))


## 5. Treatment effects and confidence intervals

In [ ]:
def welch_diff_ci(x, y, alpha=0.05):
    x, y = np.asarray(x, float), np.asarray(y, float)
    diff = x.mean() - y.mean()
    se = np.sqrt(x.var(ddof=1)/len(x) + y.var(ddof=1)/len(y))
    df = (x.var(ddof=1)/len(x) + y.var(ddof=1)/len(y))**2 / (
        (x.var(ddof=1)/len(x))**2/(len(x)-1) +
        (y.var(ddof=1)/len(y))**2/(len(y)-1)
    )
    crit = stats.t.ppf(1-alpha/2, df)
    return diff, diff-crit*se, diff+crit*se

for metric in ["retained_within_60d","revenue_per_visit_inr","no_show_flag"]:
    print(metric, welch_diff_ci(t[metric], c[metric]))


## 6. Historical difference-in-differences

In [ ]:
appointments["treated"] = (appointments.region == "Treatment Region").astype(int)
appointments["post"] = (appointments.appointment_date >= "2026-07-01").astype(int)
appointments["month"] = appointments.appointment_date.dt.to_period("M").astype(str)
outlet_month = appointments.groupby(
    ["outlet_id","region","month"], as_index=False
).agg(
    n=("appointment_id","size"),
    revenue_per_scheduled_appointment=("realized_revenue_inr","mean"),
    no_show_rate=("no_show_flag","mean")
)
outlet_month["treated"] = (outlet_month.region == "Treatment Region").astype(int)
outlet_month["post"] = (outlet_month.month >= "2026-07").astype(int)
outlet_month["did"] = outlet_month.treated * outlet_month.post

did = smf.wls(
    "revenue_per_scheduled_appointment ~ treated*post + C(outlet_id) + C(month)",
    data=outlet_month, weights=outlet_month.n
).fit(cov_type="cluster", cov_kwds={"groups":outlet_month.outlet_id})
print(did.summary())
print("DiD estimate and 95% CI:", did.params["treated:post"], did.conf_int().loc["treated:post"].tolist())


## 7. Parallel-trends diagnostic

In [ ]:
pre = outlet_month[outlet_month.month < "2026-07"].copy()
pre["month_index"] = pd.PeriodIndex(pre.month, freq="M").astype(int)
pretrend = smf.wls(
    "revenue_per_scheduled_appointment ~ treated*month_index + C(outlet_id)",
    data=pre, weights=pre.n
).fit(cov_type="cluster", cov_kwds={"groups":pre.outlet_id})
print("Pre-trend interaction:", pretrend.params["treated:month_index"])
print("p-value:", pretrend.pvalues["treated:month_index"])
# Plot raw trends; visual inspection complements, but does not prove, the assumption.
trend = outlet_month.groupby(["region","month"],as_index=False).apply(
    lambda x: pd.Series({"metric":np.average(x.revenue_per_scheduled_appointment,weights=x.n)})
).reset_index(drop=True)
for region, g in trend.groupby("region"):
    plt.plot(g.month, g.metric, marker="o", label=region)
plt.axvline(5.5, linestyle="--")
plt.xticks(rotation=45); plt.legend(); plt.ylabel("INR per scheduled appointment"); plt.tight_layout()


## 8. Segment analysis

In [ ]:
experiment["visit_segment"] = pd.cut(
    experiment.prior_visits_90d, [-1,0,2,5,100],
    labels=["New (0)","Low (1-2)","Medium (3-5)","High (6+)"]
)
segment_results = experiment.groupby(
    ["visit_segment","experiment_group"], observed=True
).retained_within_60d.agg(["mean","count"]).unstack()
display(segment_results)


## 9. Regression adjustment (CUPED-style)

In [ ]:
# Pre-treatment covariates improve precision when predictive of the outcome.
adjusted = smf.ols(
    "retained_within_60d ~ C(experiment_group) + prior_visits_90d + prior_revenue_90d_inr + age",
    data=experiment
).fit(cov_type="HC3")
print(adjusted.summary())
print("Adjusted treatment effect:", adjusted.params.get("C(experiment_group)[T.Treatment]"))
print("95% CI:", adjusted.conf_int().loc["C(experiment_group)[T.Treatment]"].tolist())


## 10. Decision framework and limitations

- **GO:** primary metric lift is commercially meaningful, confidence interval supports benefit, and guardrails remain within prespecified limits.
- **NO-GO:** credible harm to retention or guardrail breach.
- **EXTEND:** interval remains too wide to distinguish meaningful benefit from harm; continue only under a prespecified sequential plan.

Causal risks: non-parallel trends, region-specific shocks, spillovers, concurrent promotions, misclassified no-shows, missing client IDs, treatment non-compliance, attrition and synthetic-data assumptions. A non-significant pre-trend test does not prove parallel trends.
